# 03 AI Detection Synthetic Stress Test

This notebook demonstrates how a university could audit an AI detection / plagiarism tool.

The dataset is synthetic and must not be described as real student data.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
RAW_SYNTHETIC_DIR = REPO_ROOT / "data" / "raw" / "synthetic"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

input_path = RAW_SYNTHETIC_DIR / "synthetic_ai_detection_demo.csv"
print("Input:", input_path)
print("Available:", input_path.exists())

In [ ]:
if input_path.exists():
    df = pd.read_csv(input_path)
else:
    raise FileNotFoundError("Place synthetic_ai_detection_demo.csv in data/raw/synthetic/")

print(df.shape)
df.head()

In [ ]:
def yes_no_to_bool(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "yes": True,
            "no": False,
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
        .fillna(False)
        .astype(bool)
    )

audit = df.copy()

for col in [
    "actual_ai_use",
    "detection_flag",
    "human_review_completed",
    "misconduct_case_opened",
    "appeal_submitted",
    "appeal_successful",
]:
    audit[col] = yes_no_to_bool(audit[col])

audit["detector_score"] = pd.to_numeric(audit["detector_score"], errors="coerce").fillna(0)

audit["is_false_positive"] = audit["detection_flag"] & ~audit["actual_ai_use"]
audit["is_false_negative"] = ~audit["detection_flag"] & audit["actual_ai_use"]
audit["is_true_positive"] = audit["detection_flag"] & audit["actual_ai_use"]
audit["is_true_negative"] = ~audit["detection_flag"] & ~audit["actual_ai_use"]
audit["case_escalated_after_flag"] = audit["detection_flag"] & audit["misconduct_case_opened"]
audit["appeal_success_after_case"] = audit["appeal_submitted"] & audit["appeal_successful"]

audit["detector_risk_band"] = pd.cut(
    audit["detector_score"],
    bins=[-0.001, 0.30, 0.50, 0.70, 1.00],
    labels=["Low", "Borderline", "High", "Very High"],
)

audit.to_csv(PROCESSED_DIR / "ai_detection_audit_ready.csv", index=False)
audit.head()

## Fairness Metrics by Gender

Core audit metrics:

- detection flag rate
- false positive rate
- false negative rate
- human review rate
- misconduct case rate
- appeal success rate

In [ ]:
summary = (
    audit.groupby("gender", dropna=False)
    .agg(
        student_count=("student_id", "count"),
        actual_ai_use_rate=("actual_ai_use", "mean"),
        detection_flag_rate=("detection_flag", "mean"),
        false_positive_rate=("is_false_positive", "mean"),
        false_negative_rate=("is_false_negative", "mean"),
        human_review_rate=("human_review_completed", "mean"),
        misconduct_case_rate=("misconduct_case_opened", "mean"),
        appeal_submission_rate=("appeal_submitted", "mean"),
        appeal_success_rate=("appeal_successful", "mean"),
        avg_detector_score=("detector_score", "mean"),
    )
    .reset_index()
)

rate_cols = [
    "actual_ai_use_rate",
    "detection_flag_rate",
    "false_positive_rate",
    "false_negative_rate",
    "human_review_rate",
    "misconduct_case_rate",
    "appeal_submission_rate",
    "appeal_success_rate",
]

for col in rate_cols:
    summary[col] = (summary[col] * 100).round(2)

summary["avg_detector_score"] = summary["avg_detector_score"].round(3)

summary.to_csv(PROCESSED_DIR / "ai_detection_fairness_summary.csv", index=False)
summary

In [ ]:
gap_rows = []
for col in [
    "detection_flag_rate",
    "false_positive_rate",
    "false_negative_rate",
    "human_review_rate",
    "misconduct_case_rate",
    "appeal_success_rate",
]:
    gap_rows.append({
        "metric": col,
        "largest_gender_gap_percentage_points": round(summary[col].max() - summary[col].min(), 2)
    })

fairness_gaps = pd.DataFrame(gap_rows)
fairness_gaps

In [ ]:
plot_df = summary.sort_values("false_positive_rate", ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(plot_df["gender"], plot_df["false_positive_rate"])
plt.title("Synthetic AI Detection Audit: False Positive Rate by Gender")
plt.xlabel("Gender")
plt.ylabel("False positive rate (%)")
plt.tight_layout()
plt.show()



> This synthetic stress test demonstrates the audit structure a university would need to check whether AI detection tools produce unequal false positive rates across gender groups.